# Grid Search Experiment

This notebook runs a controlled grid search for the triangle-based evolutionary algorithm.

The grid enumerates every setup in the current `SEARCH_SPACE`, then runs 5 deterministic trials per setup. With the current grid, this is 12 setups x 5 trials = 60 expected runs.

The project minimizes fitness: lower RMSE is better. Fitness sharing is fixed on for every run as the baseline diversity-preservation mechanism. Restricted mating is part of the grid, so the experiment tests whether different mating restrictions add value when fitness sharing is already active.

The expensive execution cell is guarded by `RUN_EXPERIMENT = False`. When enabled, it saves `results/grid_search_raw_results.csv` after every completed trial so interrupted runs can resume safely.


In [8]:
%config InlineBackend.figure_format = 'retina'

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src import load_image
from src.ga import cross_over, fitness, grid_search, mutate

## Project API Connection

Reusable grid-search mechanics live in `src/ga/grid_search.py`. This notebook keeps only the experiment configuration, execution guard, and analysis cells.


In [ ]:
FITNESS_FUNCTION = fitness.compute_rmse
MUTATION_FUNCTION = mutate.random_triangle_mutation
CROSSOVER_FUNCTIONS = {
    "two_point_one_child": cross_over.two_point_crossover,
    "two_point_two_children": cross_over.two_point_crossover_two_children,
    "pmx": cross_over.pmx_crossover,
}

## Fixed Parameters

These values stay constant for every run. The alpha range uses the current fixed-alpha setting already used in the project notebooks.


In [10]:
BASE_SEED = 73
N_TRIALS = 5
FITNESS_OBJECTIVE = "minimize"

RESULTS_DIR = project_root / "results"
RAW_RESULTS_PATH = RESULTS_DIR / "grid_search_raw_results.csv"
SUMMARY_PATH = RESULTS_DIR / "grid_search_summary.csv"
IMAGE_PATH = project_root / "images" / "girl_pearl_earing.png"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FIXED_PARAMS = {
    "elitism": 5,
    "population_size": 250,
    "generations": 500,
    "triangle_alpha_range": (255, 255),
    "fitness_sharing": True,
    "sigma_share": 0.3,
    "n_bins": 8,
    "candidate_pool": 5,
}

FIXED_PARAMS

{'elitism': 5,
 'population_size': 250,
 'generations': 500,
 'triangle_alpha_range': (255, 255),
 'fitness_sharing': True,
 'sigma_share': 0.3,
 'n_bins': 8,
 'candidate_pool': 5}

## Search Space

The grid search evaluates every combination in this space.


In [11]:
SEARCH_SPACE = {
    "mutation_rate": [0.1],
    "crossover_rate": [0.9],
    "crossover_type": [
        "two_point_one_child",
        "two_point_two_children",
        "pmx",
    ],
    "selection_type": ["tournament", "ranking"],
    "restricted_mating": ["best_partial_match", "unidirectional"],
}

SEARCH_SPACE

{'mutation_rate': [0.1],
 'crossover_rate': [0.9],
 'crossover_type': ['two_point_one_child', 'two_point_two_children', 'pmx'],
 'selection_type': ['tournament', 'ranking'],
 'restricted_mating': ['best_partial_match', 'unidirectional']}

## Experiment Initialization

Build the complete Cartesian-product grid of experimental configurations, initialize reproducible setup ordering and trial seeds, load the target image, and prepare the result schema used throughout the experiment pipeline.

setup_id values are assigned sequentially according to the deterministic grid order, while trial seeds follow: `seed = 73 + setup_id * 100 + trial_id`


In [ ]:
# --- Generate full Cartesian-product grid of experimental configurations and load target array --- #
grid_setups = grid_search.build_grid_setups(SEARCH_SPACE)
EXPECTED_RUNS = len(grid_setups) * N_TRIALS
RESULT_COLUMNS = grid_search.RESULT_COLUMNS

# Load target array
target_array = load_image.load_target_image(IMAGE_PATH)

print(
    f"Target array: {target_array.shape} -> (H, W, 3) array with RGB values in [0, 255]"
)
print(f"Built {len(grid_setups)} grid setups for {EXPECTED_RUNS} expected trial runs.")

Target array: (400, 300, 3) -> (H, W, 3) array with RGB values in [0, 255]
Built 12 grid setups for 60 expected trial runs.


## Run Experiment

Change `RUN_EXPERIMENT` to `True` when you are ready to launch the full sequential run. The expected total run count is computed from the grid size.


In [13]:
RUN_EXPERIMENT = True

raw_results = grid_search.load_raw_results(RAW_RESULTS_PATH)
completed = (
    set(zip(raw_results["setup_id"].astype(int), raw_results["trial_id"].astype(int)))
    if not raw_results.empty
    else set()
)
expected_pairs = {
    (int(setup_id), trial_id)
    for setup_id in grid_setups["setup_id"]
    for trial_id in range(1, N_TRIALS + 1)
}

if RUN_EXPERIMENT:
    for _, setup in grid_setups.iterrows():
        setup_id = int(setup["setup_id"])
        for trial_id in range(1, N_TRIALS + 1):
            pair = (setup_id, trial_id)
            if pair in completed:
                print(f"Skipping setup {setup_id}, trial {trial_id}: already saved.")
                continue

            print(
                f"Running setup {setup_id}/{len(grid_setups)}, trial {trial_id}/{N_TRIALS}..."
            )
            result = grid_search.run_one_trial(
                setup=setup,
                trial_id=trial_id,
                target_array=target_array,
                fixed_params=FIXED_PARAMS,
                base_seed=BASE_SEED,
                fitness_function=FITNESS_FUNCTION,
                mutation_function=MUTATION_FUNCTION,
                crossover_functions=CROSSOVER_FUNCTIONS,
            )
            raw_results = pd.concat(
                [raw_results, pd.DataFrame([result])], ignore_index=True
            )
            grid_search.save_raw_results(raw_results, RAW_RESULTS_PATH)
            completed.add(pair)

            print(
                f"Saved setup {setup_id}, trial {trial_id}: "
                f"fitness={result['final_best_fitness']:.6f}, "
                f"runtime={result['runtime_seconds']:.1f}s"
            )
else:
    print("RUN_EXPERIMENT is False. Set it to True to run or resume the experiment.")

completed_pairs = expected_pairs & completed
print(f"Completed {len(completed_pairs)} / {EXPECTED_RUNS} expected runs.")

if not raw_results.empty:
    display(raw_results.sort_values(["setup_id", "trial_id"]).tail())

Running setup 1/12, trial 1/5...
Saved setup 1, trial 1: fitness=0.254734, runtime=367.5s
Running setup 1/12, trial 2/5...


KeyboardInterrupt: 

## Aggregate Results

The summary is sorted for the project's objective direction. Because RMSE fitness is minimized, lower mean final fitness is better.


In [ ]:
raw_results = grid_search.load_raw_results(RAW_RESULTS_PATH)

if raw_results.empty:
    summary = pd.DataFrame()
    print(
        "No grid search results found yet. Set RUN_EXPERIMENT = True and run the experiment cell."
    )
else:
    summary = grid_search.build_summary(raw_results)
    summary.to_csv(SUMMARY_PATH, index=False)
    print(f"Saved summary to {SUMMARY_PATH}")
    display(summary.head(10))

## Plots

These plots are conditional: they render only after raw results exist.


In [ ]:
raw_results = grid_search.load_raw_results(RAW_RESULTS_PATH)

if raw_results.empty:
    print(
        "No grid search results found yet. Set RUN_EXPERIMENT = True and run the experiment cell."
    )
else:
    summary = grid_search.build_summary(raw_results)
    plot_summary = summary.sort_values("mean_final_best_fitness", ascending=True).copy()
    plot_summary["setup_label"] = "S" + plot_summary["setup_id"].astype(str)

    runtime_summary = summary.sort_values("setup_id", ascending=True).copy()
    runtime_summary["setup_label"] = "S" + runtime_summary["setup_id"].astype(str)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Grid Search Results", fontweight="bold")

    axes[0, 0].bar(
        plot_summary["setup_label"],
        plot_summary["mean_final_best_fitness"],
        color="steelblue",
        edgecolor="white",
    )
    axes[0, 0].set_title("Mean Fitness by Setup")
    axes[0, 0].set_xlabel("Setup, sorted by mean fitness")
    axes[0, 0].set_ylabel("Mean final best fitness")
    axes[0, 0].tick_params(axis="x", rotation=90)

    raw_results.boxplot(
        column="final_best_fitness",
        by="crossover_type",
        ax=axes[0, 1],
        grid=False,
        rot=20,
    )
    axes[0, 1].set_title("Fitness Distribution by Crossover Type")
    axes[0, 1].set_xlabel("Crossover type")
    axes[0, 1].set_ylabel("Final best fitness")

    raw_results.boxplot(
        column="final_best_fitness",
        by="selection_type",
        ax=axes[1, 0],
        grid=False,
    )
    axes[1, 0].set_title("Fitness Distribution by Selection Type")
    axes[1, 0].set_xlabel("Selection type")
    axes[1, 0].set_ylabel("Final best fitness")

    axes[1, 1].bar(
        runtime_summary["setup_label"],
        runtime_summary["mean_runtime_seconds"],
        color="darkorange",
        edgecolor="white",
    )
    axes[1, 1].set_title("Runtime by Setup")
    axes[1, 1].set_xlabel("Setup")
    axes[1, 1].set_ylabel("Mean runtime seconds")
    axes[1, 1].tick_params(axis="x", rotation=90)

    plt.suptitle("Grid Search Results", fontweight="bold")
    plt.tight_layout()
    plt.show()